# Projeto: Leitura Inteligente de Tabelas – Função `ler_tabelas`

## Descrição Geral
Este notebook define uma função utilitária genérica chamada `ler_tabelas`, projetada para flexibilizar a leitura de dados em diversos formatos no ambiente Spark. A função encapsula a lógica necessária para lidar tanto com tabelas Delta quanto com arquivos CSV, Parquet e outros formatos suportados. Com múltiplos parâmetros de controle, a função permite o uso de esquemas explícitos ou inferência automática, ajustando-se a diferentes contextos de ingestão de dados em pipelines.

## Objetivos

- Centralizar a lógica de leitura de arquivos e tabelas Delta em uma função reutilizável.
- Padronizar o uso de codificação e delimitadores.
- Suportar leitura com e sem inferência de schema.
- Facilitar a leitura de dados em diferentes formatos (`csv`, `parquet`, `delta`, etc).
- Melhorar a legibilidade e evitar duplicação de lógica em notebooks de ETL.

## Tecnologias Utilizadas

- Apache Spark (PySpark)
- Delta Lake
- Databricks Notebooks

### Histórico de Atualizações

| Data       | Responsável  | Versão | Descrição                                      |
|------------|--------------|--------|-----------------------------------------------|
| 12/05/2025 | Lucas Zorn  | 1.0    | Criação inicial das funções de tratamento     |
| 19/05/2025 | Evelise Dias  | 1.2  | Inclusão de documentação    |

In [0]:
def ler_tabelas(path, schema=None, format="delta", inferSchema=False, fromTable=True, delimiter=";", encoding= "utf-8"):
    """
    Lê uma tabela Delta ou um arquivo (CSV, Parquet, etc.) e retorna um DataFrame do Spark.

    Args:
        path (str): Caminho do arquivo ou nome da tabela Delta.
        schema (StructType, optional): Esquema a ser utilizado ao ler o arquivo (caso `inferSchema` seja False).
        format (str): Formato do arquivo a ser lido (por exemplo, 'csv', 'parquet', 'delta'). Padrão é 'delta'.
        inferSchema (bool): Se True, infere automaticamente o schema do arquivo. Ignorado ao ler de tabela.
        fromTable (bool): Se True, lê diretamente de uma tabela Delta pelo nome; se False, lê de um caminho.
        delimiter (str): Delimitador dos campos no arquivo. Padrão é ','.

    Returns:
        DataFrame: DataFrame carregado a partir do caminho ou tabela fornecidos.
    """
    if fromTable:
        return spark.read.format("delta").option("encoding",encoding).table(path)
    else:
        reader = (
            spark.read.format(format)
            .option("header", True)
            .option("encoding",encoding)
            .option("delimiter", delimiter)
        )
        if inferSchema:
            return reader.option("inferSchema", True).load(path)
        elif schema is not None:
            return reader.schema(schema).load(path)
        else:
            return reader.load(path)